In [ ]:
root = "M://Diploma/experiments/vesuvius/fragments"
from utils import estimate_quality, plot_prediction

### Data & Features

For the baseline, we will do simple per-pixel classification. That is, we do not account for the neighborhood in any way. The only input we use are the raw pixel values from IR/CT.

Since fitting all 65 CT layers into RAM is infeasible (takes up more than 100 GB), here we will also select the most useful of those for further experiments, so that we do not have to worry about implementing models in external memory.

In [ ]:
import tqdm
import numpy
import typing
import warnings
import PIL.Image
import skimage.util
import scipy.ndimage
import skimage.feature
import skimage.filters

warnings.filterwarnings("ignore", category=PIL.Image.DecompressionBombWarning)

def feature_names(depths, neighborhood):
    spatial_names = ["mean", "edge", "hessian_max", "hessian_min"]
    layers = ["IR"] if depths is None else [f"CT_{depth}" for depth in depths]
    depth_names = ["depth_mean", "depth_std", "depth_range", "depth_slope", "depth_change"]
    names = layers + [f"{layer}_{name}_s{neighborhood}" for layer in layers for name in spatial_names]
    return names if depths is None else names + depth_names

def depth_std(profile):
    mean = numpy.mean(profile, axis=1, dtype=numpy.float64)
    variance = numpy.einsum("ij,ij->i", profile, profile, dtype=numpy.float64) / profile.shape[1] - mean ** 2
    return numpy.sqrt(numpy.maximum(variance, 0))

def depth_slope(profile, depths):
    centered = numpy.array(depths) - numpy.mean(depths)
    return numpy.einsum("ij,j->i", profile, centered / max(numpy.sum(centered ** 2), 1))

def depth_change(profile):
    change = numpy.zeros(len(profile), dtype=numpy.float32)
    for i in range(profile.shape[1] - 1):
        change += numpy.abs(profile[:, i + 1] - profile[:, i])
    return change / max(profile.shape[1] - 1, 1)

def curvature(smoothed):
    hessian = skimage.feature.hessian_matrix(smoothed, sigma=0, use_gaussian_derivatives=False)
    return skimage.feature.hessian_matrix_eigvals(hessian)

def load_fragment(fragment: int, depths: typing.Optional[typing.List[int]], neighborhood):
    mask = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/mask.png")) > 0
    pixels = numpy.flatnonzero(mask)
    ink = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/inklabels.png")).ravel()[pixels] > 0

    filenames = ["ir.png"] if depths is None else [f"surface_volume/{depth:02d}.tif" for depth in depths]
    features = numpy.empty((len(pixels), len(feature_names(depths, neighborhood))), dtype=numpy.float32)

    nearest = scipy.ndimage.distance_transform_edt(~mask, return_distances=False, return_indices=True)
    for i, filename in enumerate(tqdm.tqdm(filenames)):
        image = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/{filename}"))
        features[:, i] = image.ravel()[pixels]

        offset = len(filenames) + 4 * i
        smoothed = skimage.filters.gaussian(
            skimage.util.img_as_float32(image[tuple(nearest)]), sigma=neighborhood
        )
        features[:, offset] = smoothed.ravel()[pixels]
        features[:, offset + 1] = skimage.filters.sobel(smoothed).ravel()[pixels]
        largest, smallest = curvature(smoothed)
        features[:, offset + 2] = largest.ravel()[pixels]
        features[:, offset + 3] = smallest.ravel()[pixels]

    if depths is not None:
        profile = features[:, :len(depths)]
        features[:, -5] = numpy.mean(profile, axis=1, dtype=numpy.float64)
        features[:, -4] = depth_std(profile)
        features[:, -3] = numpy.ptp(profile, axis=1)
        features[:, -2] = depth_slope(profile, depths)
        features[:, -1] = depth_change(profile)
    return features, ink

def load_fragments(fragments, depths: typing.Optional[typing.List[int]], neighborhood):
    values = list(load_fragment(fragments[0], depths, neighborhood))
    for fragment in fragments[1:]:
        fragment_values = load_fragment(fragment, depths, neighborhood)
        for i in range(len(values)):
            values[i] = numpy.concatenate([ values[i], fragment_values[i] ])
    return values

### Model

In [ ]:
import matplotlib.pyplot as plt

def draw_feature_importance(importances, feature_names, title):
    print('Most important:', numpy.array(feature_names)[numpy.argsort(numpy.abs(importances))[::-1]])
    plt.figure(figsize=(6, 2))
    plt.plot(feature_names, importances)
    plt.xlabel("Feature index (layer)")
    plt.ylabel("Importance")
    plt.title(title)
    plt.show()

In [ ]:
import pandas
import catboost
import sklearn.preprocessing

def run_experiment(depths, neighborhood):
    train_X, train_y = load_fragments([1, 2, 3], depths=depths, neighborhood=neighborhood)
    test_X, test_y = load_fragments([4], depths=depths, neighborhood=neighborhood)

    ss = sklearn.preprocessing.StandardScaler(copy=False)
    train_X = ss.fit_transform(train_X)
    test_X = ss.transform(test_X)

    model = catboost.CatBoostClassifier(
        class_weights=[1, 5], iterations=100, depth=4, random_seed=42
    )
    model = model.fit(train_X, train_y)

    display(estimate_quality(model.predict_proba(train_X)[:, 1], train_y))

    preds = model.predict_proba(test_X)[:, 1]
    display(estimate_quality(preds, test_y))
    plot_prediction(preds, fragment=4, ax=None)

    scores = pandas.DataFrame({"feature name": feature_names(None, 3), "importance": model.feature_importances_})
    scores = scores.transpose().sort_values(by='importance')
    display(scores.head(10))
    display(scores.tail(10))

#### IR

In [ ]:
model = run_experiment(None, 3)

In [ ]:
run_experiment(None, 5)

In [ ]:
run_experiment(None, 7)

In [ ]:
run_experiment(None, 13)

In [ ]:
run_experiment(None, 21)

In [ ]:
run_experiment(None, 30)

In [ ]:
run_experiment(None, 50)

#### CT

In [ ]:
run_experiment(list(range(15, 46, 3)), 3)

In [ ]:
run_experiment(list(range(15, 46, 3)), 5)

In [ ]:
run_experiment(list(range(15, 46, 3)), 7)

In [ ]:
run_experiment(list(range(15, 46, 3)), 13)

In [ ]:
run_experiment(list(range(15, 46, 3)), 21)

In [ ]:
run_experiment(list(range(15, 46, 3)), 30)

In [ ]:
run_experiment(list(range(15, 46, 3)), 50)

#### Results

Expectedly, infrared scans provide a significantly better opportunity for good predictions, with all models exceeding 0.31 $F_{0.5}$ score except for the logistic regression.

CT scans, however, show much more depressing results. Even the best models only barely improve upon the dummy classifier that predicts inc in every pixel ($F_{0.5} = 0.12649$). The best results from each section are the following:
```
CatBoost	Every_2nd	0.134024
CatBoost	Every_3rd	0.133919
CatBoost	Every_4th	0.133222
Gradient boosting	Inner	0.132396
CatBoost	Middle	0.134733
Decision stump	Outer	0.126542
```
Expectedly, middle layers show the best results - even better than selecting every 2nd layer. Therefore, our assumptions about layer importances are probably correct. This is also supported by the analyses of the models: both logistic regression coefficients and tree/forest/boosting feature importances have a clear peak around layers 20-40.

Before going into deep learning, in the further experiments, we will employ catboost using midline layers from 15 to 45 with a stride of 2. This should be a reasonable balance between quality, memory usage, and speed.

#### Best result

Well... Kinda terrible